# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/james-kalaw/FlyRank-ML-Intern-Kalaw/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: signal analysis.** It is not one of the four types above, and that is on purpose.

The framing skill lists four task types. My question ("which common SEO beliefs hold up in this data?") matches the fourth one, *"Which signals travel together?" → signal analysis*: no trained model, measured with effect sizes and grouped comparisons.

It is *supervised in a loose sense*: every signal is compared against the same observed outcome (whether a page is declining). But I am not building a classifier that predicts decline for new pages, and I am not ranking pages. The output is a **verdict per signal**, so it is closest to *scoring signals*, not scoring pages.

Why not the others:
- **Classification / ranking:** those answer "which pages will decline / which first?" (Lane 2). My customer needs "which beliefs are true?", which a page-level score doesn't answer.
- **Clustering:** finds page types with no outcome at all (Lane 3). I need an outcome to judge each belief against.

The code below loads the data and builds the outcome column used in every section.

In [ ]:
# Setup: load the starter CSV (clones my repo first if the data folder is missing)
import os, subprocess
import numpy as np
import pandas as pd

DATA = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(DATA):
    if not os.path.isdir("repo"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/james-kalaw/FlyRank-ML-Intern-Kalaw.git", "repo"],
                       check=True)
    os.chdir("repo")

df = pd.read_csv(DATA)

# The outcome every signal is compared against
df["is_declining"] = (df["trend_direction"] == "down").astype(int)
BASE_RATE = df["is_declining"].mean() * 100

print(f"{len(df):,} pages, {df['client_id'].nunique()} clients")
print(f"Base rate: {BASE_RATE:.1f}% of pages are declining")

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Outcome (proxy): `is_declining` = 1 when `trend_direction == "down"`, else 0.** Base rate: about 54% of pages.

**Observed or defined?** Both, in two layers, and I want to be honest about it:
1. **Observed:** `trend_pct`, the measured % change in Google impressions between the last 30 days and the 30 days before (`(last30 − prev30) / prev30 × 100`).
2. **Defined by a rule:** `trend_direction` cuts that number into buckets. Below −20% is `down`.

So the label is a **rule applied to an observed measurement**, not a rule someone invented about what *should* decline. That's acceptable for a signal report, but it means the −20% cut-off is a choice. A page at −19% counts as "not declining." The code below checks the cut-off and that `trend_pct` really comes from the impression windows.

**Two limits I'm accepting for now:**
- **Same time window.** In the starter CSV, the signals (e.g. `impressions_90d`, `ctr`) and the outcome come from the *same* 90 days. My results will show which signals **go together** with decline, not which ones **come before** it. The Week 3 warehouse has daily data, so I can measure signals in an earlier window and decline in a later one.
- **Columns I will never use as signals**, because they are the outcome or its ingredients: `trend_direction`, `trend_pct`, and the six `*_last_30d` / `*_prev_30d` columns. `impressions_90d` also overlaps the outcome window, so any result on it gets a warning label.

In [ ]:
# Where the label comes from

# 1. The cut-off: trend_pct range inside each trend_direction bucket
print("trend_pct (%) by trend_direction:")
print(df.groupby("trend_direction")["trend_pct"].agg(["count", "min", "max"]).round(1).to_string())

# 2. trend_pct really is the change between the two 30-day impression windows
prev = df["impressions_prev_30d"]
last = df["impressions_last_30d"]
has_prev = prev > 0
recomputed = (last - prev) / prev * 100
match = ((recomputed - df["trend_pct"]).abs() <= 0.1)[has_prev].mean() * 100
print(f"\ntrend_pct matches (last30 - prev30) / prev30 on {match:.1f}% of pages with prev30 > 0")
print(f"Pages with prev30 = 0 (trend_pct blank, 'new' or 'flat'): {(~has_prev).sum():,}")

# 3. Columns that are the outcome or its ingredients: never used as signals
LABEL_COLS = ["trend_direction", "trend_pct", "is_declining"]
LABEL_INGREDIENTS = [c for c in df.columns if c.endswith("_last_30d") or c.endswith("_prev_30d")]
print("\nNever used as signals:", LABEL_COLS + LABEL_INGREDIENTS)

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: the decline-rate gap.** For one signal at a time:

> **gap = (decline rate among the top 25% of pages on that signal) − (decline rate among the bottom 25%)**, in percentage points.

Example: if the longest 25% of pages decline at 50% and the shortest 25% at 58%, the gap is **−8 points**. Longer pages decline *less*.

**What counts as "good" (decided now, before looking at most signals):**

| Gap size | Verdict |
|---|---|
| 5 points or more (either sign) | **notable**: worth a closer look |
| 2 to under 5 points | **weak** |
| under 2 points | **none**: the belief doesn't show up |

These are in absolute percentage points against a ~54% base rate. A notable signal also needs at least ~500 pages in each group, and later it must keep its direction when I check it client by client (Week 6), so it isn't driven by one or two clients.

**Why this metric:** it's in plain English ("pages with lots of X decline 8 points less"), it's a grouped comparison (which caught the CTR cliff that a correlation missed), and it shows the base rate right next to it.

**Known weaknesses:** comparing only the two ends ignores the middle 50%, so a U-shape (both ends high) can show a gap near zero. I'll catch that by also looking at tier tables (Section 5). Columns with many zeros (like `search_volume`) can give unequal group sizes, so the code prints the group sizes.

The code runs the metric on four signals I already looked at in Week 1, as a first baseline run.

**Result (fill in after Run all):** _write the four verdicts here in one or two sentences._

In [ ]:
# The metric: decline-rate gap between the top and bottom 25% of a signal

def decline_gap(data, signal, outcome="is_declining"):
    d = data[[signal, outcome]].dropna()
    low_cut, high_cut = d[signal].quantile([0.25, 0.75])
    bottom = d.loc[d[signal] <= low_cut, outcome]
    top = d.loc[d[signal] >= high_cut, outcome]
    gap = (top.mean() - bottom.mean()) * 100
    verdict = "notable" if abs(gap) >= 5 else ("weak" if abs(gap) >= 2 else "none")
    return {"signal": signal,
            "pages_bottom": len(bottom), "pages_top": len(top),
            "bottom_decline_%": round(bottom.mean() * 100, 1),
            "top_decline_%": round(top.mean() * 100, 1),
            "gap_pts": round(gap, 1), "verdict": verdict}

# avg_position = 0 means "no data", not rank zero, so drop those pages for that signal
with_position = df[df["avg_position"] > 0]

rows = [decline_gap(df, "content_age_days"),
        decline_gap(df, "word_count"),
        decline_gap(df, "search_volume"),
        decline_gap(with_position, "avg_position")]
print(f"Base rate: {BASE_RATE:.1f}% declining\n")
print(pd.DataFrame(rows).set_index("signal").to_string())

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one page** (one content item), with its metrics over a trailing 90-day window. Pages belong to clients (websites), and a client has many pages.

The code checks this instead of assuming it:
- every `content_id` appears exactly once, so one row really is one page;
- how many pages each client has, because a few big clients could dominate any average;
- the **role of every column**: ID (grouping only), outcome, outcome ingredient (excluded), metadata (excluded per the data dictionary), or **signal** (allowed to be tested). It also shows how much of each is missing.

I'm showing the table's structure, not raw rows, because the rows aren't needed to see the shape. That also keeps row-level data out of this notebook's outputs.

**Result (fill in after Run all):** _how many signals are there, and which ones have a lot missing?_

In [ ]:
# Unit of analysis: one row = one page

print(f"Rows: {len(df):,} | unique content_id: {df['content_id'].nunique():,}")
print("One row per page:", df["content_id"].is_unique)

pages_per_client = df.groupby("client_id").size()
print(f"\nPages per client: min {pages_per_client.min():,}, "
      f"median {int(pages_per_client.median()):,}, max {pages_per_client.max():,}")
print(f"Biggest 3 clients hold {pages_per_client.nlargest(3).sum() / len(df):.1%} of all pages")

# Role of every column
ID_COLS = ["content_id", "client_id"]
METADATA_EXCLUDED = ["provider_used", "model_used"]  # data dictionary: "Not a model feature"
role = {}
for c in df.columns:
    if c in ID_COLS: role[c] = "id (grouping only)"
    elif c in LABEL_COLS: role[c] = "outcome"
    elif c in LABEL_INGREDIENTS: role[c] = "outcome ingredient (excluded)"
    elif c in METADATA_EXCLUDED: role[c] = "metadata (excluded)"
    else: role[c] = "signal"

columns = pd.DataFrame({"role": pd.Series(role),
                        "dtype": df.dtypes.astype(str),
                        "missing_%": (df.isna().mean() * 100).round(1)})
columns = columns.sort_values(["role", "missing_%"], ascending=[True, False])
print("\nColumns by role:")
print(columns["role"].value_counts().to_string())
print()
print(columns.to_string())

SIGNALS = columns.index[columns["role"] == "signal"].tolist()

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Honest answer: in my lane, I'm not claiming ML beats a rule.** Lane 1 doesn't train a model. The framing skill says ML only earns its place when the pattern is real but too messy to write by hand. What I can show is that **the folklore rules teams already use** (simple if-statements like "old pages decline") **are too simple for this data**. That is why a careful signal-by-signal check is needed instead of a rule of thumb.

Evidence I already have:
- **Backwards patterns:** declining pages have the *youngest* median age (216 days vs 300 for stable). The rule "old content decays" points the wrong way.
- **Non-linear patterns:** position vs CTR has almost no linear correlation (−0.073), yet CTR drops about 6× from page 1 to deep results. An if-statement with one cut-off, or a straight-line check, misses that shape.
- **Interacting signals:** in Week 2 the decision tree split on `impressions_90d` first and *then* on `content_age_days`. Age only mattered once a page had some traffic, so one signal's effect depends on another.

The code tests this directly:
1. scores a folklore rule ("pages 365+ days old are the ones declining") against the base rate;
2. shows decline rate across age tiers and position tiers, to see whether it moves in a straight line or bends;
3. crosses age tier with impression tier, to see whether the age pattern changes depending on traffic level (an interaction).

**Result (fill in after Run all):** _did the folklore rule beat the base rate? Is either tier pattern a straight line? Does the age pattern change across impression tiers?_

In [ ]:
# Why a one-line rule isn't enough: backwards, bent, and interacting patterns

def rate_by(col, order=None):
    t = df.groupby(col)["is_declining"].agg(pages="size", decline_rate="mean")
    t["decline_rate"] = (t["decline_rate"] * 100).round(1)
    if order is not None:
        t = t.reindex([o for o in order if o in t.index])
    return t

# 1. A folklore rule, scored against the base rate
old = df["content_age_days"] >= 365
print(f"Rule 'pages 365+ days old are declining': {old.sum():,} pages flagged, "
      f"{df.loc[old, 'is_declining'].mean() * 100:.1f}% of them declining "
      f"(base rate {BASE_RATE:.1f}%)")

# 2. Does decline move in a straight line across tiers?
age_order = (df[["age_tier", "age_tier_order"]].drop_duplicates()
             .sort_values("age_tier_order")["age_tier"].tolist())
pos_order = ["top_3", "page_1", "striking", "page_3_5", "deep", "no_data"]
print("\nDecline rate (%) by age tier:")
print(rate_by("age_tier", age_order).to_string())
print("\nDecline rate (%) by position tier:")
print(rate_by("position_tier", pos_order).to_string())

# 3. Interaction: does the age pattern change with traffic level?
imp_order = ["no_data", "none", "low", "moderate", "good", "excellent"]
rates = df.pivot_table(index="age_tier", columns="impression_tier",
                       values="is_declining", aggfunc="mean") * 100
sizes = df.pivot_table(index="age_tier", columns="impression_tier",
                       values="is_declining", aggfunc="size")
rates = rates.where(sizes >= 100)  # hide cells with fewer than 100 pages (too noisy)
rates = rates.reindex(index=age_order,
                      columns=[c for c in imp_order if c in rates.columns]).round(1)
print("\nDecline rate (%) by age tier x impression tier (blank = under 100 pages):")
print(rates.to_string())

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.